In [1]:
import string
import pandas as pd

from collections import Counter

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer


df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  A

In [2]:
# Getting the freequency distribution of the answers
counts = df["answer"].value_counts()
print(counts)

# Getting the most freequent and least freequent answer
most_freequency = counts.max()
least_freequency = counts.min()

result = most_freequency + least_freequency
print(result)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
814


In [3]:
# Make a translation table ( a mapping of characters ordinals to replacement )
# Basically replace all punchuations with nothing
translator = str.maketrans("", "", string.punctuation)

all_words = set()

# Getting each prompt at a time and performing cleanup
for text in df["prompt"]:
    cleaned = text.lower().translate(translator)
    words = cleaned.split()

    all_words.update(words)

print(len(all_words))

859


In [4]:
# Getting the Row ID = 1
row = df[df["id"] == 1]
text = row["prompt"].iloc[0]

# Using the translator form the cell above
cleaned = text.lower().translate(translator)
words = cleaned.split()

# Setting the lsit for filtered words
filtered_words = [
    word for word in words
    if word not in ENGLISH_STOP_WORDS
]

print(filtered_words)
print(len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


In [5]:
combined_text = (
    df["prompt"] + " " +
    df["A"] + " " +
    df["B"] + " " + 
    df["C"] + " " + 
    df["D"] + " " + 
    df["E"] 
)

# Converting into term freequency matrix using sklearn's TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words = 'english')
x = vectorizer.fit_transform(combined_text)

print(x.shape[1])

2762


In [6]:
# Getting the row variable from cell 4
prompt_vec = vectorizer.transform(row["prompt"])
A_vec = vectorizer.transform(row["A"])

similarity = cosine_similarity(prompt_vec, A_vec)[0, 0]

print(round(similarity, 4))

0.272


In [7]:
options = ["A", "B", "C", "D", "E"]

correct = 0

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in options:
        option_vec = vectorizer.transform([row[option]])

        # Checking the similarity between given prompt and option
        similarities[option] = cosine_similarity(prompt_vec, option_vec)[0, 0]

    prediction = max(similarities, key=similarities.get)

    if prediction == row["answer"]:
        correct += 1

accuracy = correct / len(df) * 100

print(round(accuracy, 2))

13.55


In [8]:
# Mean Average Precision @ 3
def map3_score(actual, predictions):
    for i, prediction in enumerate(predictions, start=1):
        if prediction == actual:
            return 1/i
            
    return 0

print(map3_score("C", ["C", "A", "B"]))
print(map3_score("B", ["D", "B", "E"]))

1.0
0.5


In [9]:
# Getting counts from cell 2
top3 = counts.index.tolist()[:3]
print(top3)

scores = []

for answer in df["answer"]:
    scores.append(map3_score(answer, top3))

map3 = sum(scores) / len(scores)

print(round(map3, 4))

['B', 'C', 'A']
0.4213


In [10]:
scores = []

options = ["A","B","C","D","E"]

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])
    similarities = {}

    for option in options:

        option_vec = vectorizer.transform([row[option]])
        similarities[option] = cosine_similarity(prompt_vec, option_vec)[0,0]

    top3_predictions = sorted(similarities, key=similarities.get, reverse=True)[:3]

    scores.append(
        map3_score(row["answer"], top3_predictions)
    )

final_map3 = sum(scores) / len(scores)

print(round(final_map3,4))

0.2962
